# Comparação entre IA e pipeline matemático

## Objetivo

Compara os resultados do pipeline matemático com os métodos de IA usando somente exames presentes nas duas fontes.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
from typing import Optional
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from utils.comparison_utils.ia_math import prettify_method_label

try:
    import plotly.graph_objects as go
    import plotly.express as px
except Exception:
    go = None
    px = None


def plot_comparison_bar_by_resolution(
    agg: pd.DataFrame, resolution: str, comparison_title: Optional[str] = None
) -> None:
    """Plota comparação de Dice por método para uma resolução alvo."""
    if agg.empty or "target_resolution" not in agg.columns:
        plt.figure(figsize=(10, 5))
        plt.text(0.5, 0.5, f"Sem dados para {resolution}", ha="center", va="center")
        plt.axis("off")
        plt.show()
        return

    # Filtra somente a resolução solicitada.
    subset = agg[agg["target_resolution"] == resolution].copy()
    if subset.empty:
        plt.figure(figsize=(10, 5))
        plt.text(0.5, 0.5, f"Sem dados para {resolution}", ha="center", va="center")
        plt.axis("off")
        plt.show()
        return

    subset["origin_label"] = subset["source"].map({"ia": "IA", "math": "Matematico"})
    subset["method_label"] = subset["method"].apply(prettify_method_label)
    # Ordena métodos por média de Dice.
    method_order = subset.sort_values("mean_dice", ascending=False)[
        "method_label"
    ].tolist()
    subset["method_label"] = pd.Categorical(
        subset["method_label"], categories=method_order, ordered=True
    )
    subset = subset.sort_values("method_label")

    # Plota barras separando IA e Matemático por cor.
    plt.figure(figsize=(12, 5))
    ax = sns.barplot(
        data=subset,
        x="method_label",
        y="mean_dice",
        hue="origin_label",
        palette={"IA": "#4C78A8", "Matematico": "#F58518"},
    )

    for idx, row in subset.reset_index(drop=True).iterrows():
        # Usa desvio padrão como barra de erro.
        std_val = row["std_dice"]
        if pd.notna(std_val):
            ax.errorbar(
                x=idx,
                y=row["mean_dice"],
                yerr=std_val,
                fmt="none",
                ecolor="black",
                elinewidth=1.2,
                capsize=3,
            )

    if comparison_title:
        ax.set_title(f"{comparison_title} - {resolution}")
    else:
        ax.set_title(f"Comparacao de Dice por metodo - {resolution}")
    ax.set_xlabel("Metodo")
    ax.set_ylabel("Dice medio")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=35)
    for tick_label in ax.get_xticklabels():
        tick_label.set_ha("right")

    for container in ax.containers:
        if hasattr(container, "patches") and container.patches:
            ax.bar_label(container, fmt="%.3f", padding=3)

    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Origem")
    plt.tight_layout()
    plt.show()


def plot_image_dice_scatter_interactive(
    comparison_df: pd.DataFrame, resolution: str, comparison_title: Optional[str] = None
) -> None:
    """Plota Dice interativo por imagem para IA e método matemático.

    O cursor exibe `img_id`, Dice e método.
    """
    if px is None or go is None:
        raise ImportError(
            "Plotly is required for interactive plots. Please install plotly."
        )

    subset = comparison_df[comparison_df["target_resolution"] == resolution].copy()
    if subset.empty:
        fig = go.Figure()
        title = comparison_title or "Comparacao de Dice por imagem"
        fig.add_annotation(
            text=f"Sem dados para {title} - {resolution}", showarrow=False
        )
        fig.show()
        return

    subset = subset.sort_values(["ia_method", "img_id"]).reset_index(drop=True)

    fig = go.Figure()
    colors = px.colors.qualitative.Plotly
    for method_index, (method, method_df) in enumerate(subset.groupby("ia_method")):
        method_label = prettify_method_label(method)
        fig.add_trace(
            go.Scatter(
                x=method_df["img_id"].astype(str),
                y=method_df["ia_dice"],
                mode="markers",
                marker=dict(color=colors[method_index % len(colors)], size=7),
                name=method_label,
                customdata=method_df[["img_id", "ia_method"]].values,
                hovertemplate=(
                    "img_id: %{customdata[0]}<br>Dice: %{y:.3f}"
                    "<br>Metodo: %{customdata[1]}<extra></extra>"
                ),
            )
        )

    # O Dice matemático é igual nas linhas repetidas por método de IA.
    math_df = subset.drop_duplicates(["target_resolution", "img_id"])
    fig.add_trace(
        go.Scatter(
            x=math_df["img_id"].astype(str),
            y=math_df["math_dice"],
            mode="markers",
            marker=dict(color="#222222", size=8, symbol="x"),
            name="Pipeline matemático",
            customdata=math_df[["img_id", "math_method"]].values,
            hovertemplate="img_id: %{customdata[0]}<br>Dice: %{y:.3f}<br>Metodo: %{customdata[1]}<extra></extra>",
        )
    )

    title = comparison_title or "Comparacao Dice por imagem"
    fig.update_layout(
        title=f"{title} - {resolution}",
        xaxis_title="Imagem",
        yaxis_title="Dice",
        yaxis=dict(range=[0, 1.05]),
        template="simple_white",
        legend_title_text="Método",
        margin=dict(l=40, r=20, t=60, b=120),
    )
    fig.update_xaxes(tickangle=45)
    fig.show()


def plot_ia_vs_math_scatter_interactive(
    comparison_df: pd.DataFrame, resolution: str, comparison_title: Optional[str] = None
) -> None:
    """Plota a relação interativa entre IA e método matemático.

    O cursor exibe `img_id`, métodos e valores.
    """
    if px is None:
        raise ImportError(
            "Plotly is required for interactive plots. Please install plotly."
        )

    subset = comparison_df[comparison_df["target_resolution"] == resolution].copy()
    if subset.empty:
        fig = go.Figure()
        title = comparison_title or "Comparacao IA vs Matematico"
        fig.add_annotation(
            text=f"Sem dados para {title} - {resolution}", showarrow=False
        )
        fig.show()
        return

    subset["ia_method_label"] = subset["ia_method"].apply(prettify_method_label)
    fig = px.scatter(
        subset,
        x="ia_dice",
        y="math_dice",
        hover_data=["img_id", "ia_method", "math_method"],
        color="ia_method_label",
        height=600,
        width=760,
    )
    fig.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray")
    )
    title = comparison_title or "Comparacao IA vs Matematico"
    fig.update_layout(
        title=f"{title} - {resolution}",
        legend_title_text="Método de IA",
    )
    fig.update_xaxes(range=[0, 1.05], title_text="Dice IA")
    fig.update_yaxes(range=[0, 1.05], title_text="Dice Matematico")
    fig.show()

In [ ]:
import pandas as pd

from utils.project.runtime.notebook_env import get_default_split_paths
from utils.comparison_utils.ia_math import (
    build_comparison_agg_df,
    filter_to_common_ia_math_ids,
    get_common_ia_math_keys,
    load_ia_results_for_comparison,
    load_math_results_for_comparison,
)
from utils.comparison_utils.ostia_scenarios import (
    build_ostia_image_comparison_df,
    load_ostia_comparison_scenario,
)

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="deep",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 110,
        "savefig.dpi": 300,
    },
)
plt.close("all")

SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
VALID_SPLITS = ("train", "val", "test")

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento de dados

### Dados para comparação com métodos de IA

In [ ]:
IA_RESULTS_BASE = REPO_ROOT / "output/ia_results"
MATH_TEST_SUMMARY_PATHS = {
    resolution: {
        split_name: split_path / f"results_{split_name}.csv"
        for split_name, split_path in split_paths.items()
        if split_name in VALID_SPLITS
    }
    for resolution, split_paths in SPLIT_PATHS_BY_RESOLUTION.items()
}

ia_results_df, missing_ia_files = load_ia_results_for_comparison(IA_RESULTS_BASE)
math_results_df, missing_math_files = load_math_results_for_comparison(
    MATH_TEST_SUMMARY_PATHS
)
comparison_raw_df = pd.concat([ia_results_df, math_results_df], ignore_index=True)
comparison_common_df = filter_to_common_ia_math_ids(comparison_raw_df)
comparison_agg_df = build_comparison_agg_df(comparison_common_df)
common_ids_by_resolution = (
    get_common_ia_math_keys(comparison_raw_df)
    .groupby("target_resolution")["img_id"]
    .nunique()
)

print("Shapes carregados para comparação IA:")
print("- IA bruto:", ia_results_df.shape)
print("- Matemático bruto:", math_results_df.shape)
print("- Consolidado bruto:", comparison_raw_df.shape)
print("- Consolidado com IDs comuns:", comparison_common_df.shape)
print("- Agregado:", comparison_agg_df.shape)
print("\nIDs comuns IA x Matemático por resolução:")
print(common_ids_by_resolution)

if missing_ia_files:
    print("\nAvisos IA:")
    for msg in missing_ia_files:
        print("-", msg)

if missing_math_files:
    print("\nAvisos Matemático:")
    for msg in missing_math_files:
        print("-", msg)

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Comparação: Métodos de IA vs Método Matemático

In [ ]:
def display_comparison_scenario(
    scenario_label: str,
    scenario_math_df: pd.DataFrame,
    scenario_ia_df: pd.DataFrame,
    resolution: str,
) -> None:
    # Reune IA e pipeline matematico antes de calcular as metricas do cenario.
    scenario_raw_df = pd.concat([scenario_ia_df, scenario_math_df], ignore_index=True)
    scenario_agg_df = build_comparison_agg_df(scenario_raw_df)

    # A funcao de visualizacao tambem exibe uma mensagem quando nao ha dados.
    plot_comparison_bar_by_resolution(
        scenario_agg_df,
        resolution,
        comparison_title=scenario_label,
    )

### 1. Conjunto completo de dados

In [ ]:
full_ia_df, full_math_df, full_missing_math_files = load_ostia_comparison_scenario(
    ia_results_df,
    MATH_TEST_SUMMARY_PATHS,
    "full",
)
display_comparison_scenario(
    "Completo",
    full_math_df,
    full_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Completo",
    full_math_df,
    full_ia_df,
    "high_res",
)

### 2. Apenas casos em que os óstios são encontrados corretamente

In [ ]:
correct_ia_df, correct_math_df, correct_missing_math_files = (
    load_ostia_comparison_scenario(
        ia_results_df,
        MATH_TEST_SUMMARY_PATHS,
        "correct",
    )
)
display_comparison_scenario(
    "Apenas corretos",
    correct_math_df,
    correct_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Apenas corretos",
    correct_math_df,
    correct_ia_df,
    "high_res",
)

### 3. Apenas casos em que os óstios não são encontrados corretamente

In [ ]:
incorrect_ia_df, incorrect_math_df, incorrect_missing_math_files = (
    load_ostia_comparison_scenario(
        ia_results_df,
        MATH_TEST_SUMMARY_PATHS,
        "incorrect",
    )
)
display_comparison_scenario(
    "Apenas incorretos",
    incorrect_math_df,
    incorrect_ia_df,
    "mid_res",
)

In [ ]:
display_comparison_scenario(
    "Apenas incorretos",
    incorrect_math_df,
    incorrect_ia_df,
    "high_res",
)

## Comparação por imagem interativa: IA vs método matemático

Abaixo os gráficos interativos separados por cenário e resolução. Cada método de IA é apresentado como uma série independente, sem selecionar o melhor método por exame. Passe o mouse sobre os pontos para ver `img_id`, Dice e método.

### Completo

In [ ]:
# Completo - Mid Res
scenario_image_df = build_ostia_image_comparison_df(full_ia_df, full_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Completo"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Completo"
)

In [ ]:
# Completo - High Res
scenario_image_df = build_ostia_image_comparison_df(full_ia_df, full_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Completo"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Completo"
)

### Apenas corretos

In [ ]:
# Apenas corretos - Mid Res
scenario_image_df = build_ostia_image_comparison_df(correct_ia_df, correct_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas corretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas corretos"
)

In [ ]:
# Apenas corretos - High Res
scenario_image_df = build_ostia_image_comparison_df(correct_ia_df, correct_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas corretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas corretos"
)

### Apenas incorretos

In [ ]:
# Apenas incorretos - Mid Res
scenario_image_df = build_ostia_image_comparison_df(incorrect_ia_df, incorrect_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas incorretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "mid_res", comparison_title="Apenas incorretos"
)

In [ ]:
# Apenas incorretos - High Res
scenario_image_df = build_ostia_image_comparison_df(incorrect_ia_df, incorrect_math_df)
plot_image_dice_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas incorretos"
)
plot_ia_vs_math_scatter_interactive(
    scenario_image_df, "high_res", comparison_title="Apenas incorretos"
)